# Frente 3 - Diversidade, localização e modelos de trabalho

**State of Data Brasil 2023 · 2024 · 2025-2026**.

> **A pergunta que guia tudo:** *quem tem acesso às oportunidades do mercado de
> Dados, e como essas oportunidades se distribuem?*

### De onde vêm os dados

As 4 tabelas do database `dados_gold`. Nenhum número aqui é calculado a partir da
silver.

### Onde os gráficos são salvos

Gráficos são salvos em: `s3://tech-challenge-014478672967/artefatos/frente3_diversidade_loc_modalidade/`

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%additional_python_modules matplotlib

## Configuração

In [ ]:
import io

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import boto3

from awsglue.context import GlueContext
from pyspark.context import SparkContext

spark = GlueContext(SparkContext.getOrCreate()).spark_session

BUCKET = "tech-challenge-014478672967"
PASTA_FIG = "artefatos/frente3_diversidade_loc_modalidade"
s3 = boto3.client("s3")

SURFACE, INK, INK2, MUTED, GRID, AXIS = (
    "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7")
SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # categorico
RAMPA3 = ["#86b6ef", "#2a78d6", "#104281"]                        # ordinal
RAMPA4 = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]             # ordinal

ORDEM_NIVEL = ["Júnior", "Pleno", "Sênior"]
ORDEM_MODELO = ["Remoto", "Híbrido flexível", "Híbrido fixo", "Presencial"]
ORDEM_REGIAO = ["Sudeste", "Sul", "Nordeste", "Centro-oeste", "Norte"]
ANOS = [2023, 2024, 2025]
NI = "Não informado"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.size": 10,
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans", "sans-serif"],
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "semibold",
    "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "xtick.bottom": False, "ytick.left": False,
    "legend.frameon": False, "legend.fontsize": 9,
    "lines.linewidth": 2, "lines.markersize": 7,
})


def moldura(ax, titulo, subtitulo=None, eixo="y"):
    ax.set_title(titulo, pad=26 if subtitulo else 10)
    if subtitulo:
        ax.text(0, 1.012, subtitulo, transform=ax.transAxes,
                fontsize=9.5, color=INK2, va="bottom")
    ax.grid(False)
    ax.grid(axis=eixo)
    ax.set_axisbelow(True)


def mostrar(fig, nome):
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=140, bbox_inches="tight")
    png = buf.getvalue()

    s3.put_object(Bucket=BUCKET, Key=f"{PASTA_FIG}/{nome}.png",
                  Body=png, ContentType="image/png")
    try:
        from IPython.display import Image, display
        display(Image(data=png))
    except Exception:
        pass
    plt.close(fig)
    print(f"figura: s3://{BUCKET}/{PASTA_FIG}/{nome}.png")


def tabela(df, titulo):
    print(f"\n{titulo}\n{df.to_string()}")


# --- carga da gold -------------------------------------------------------
gs, gc, rg, mt = (spark.table(f"dados_gold.{t}").toPandas() for t in
                  ["gold_genero_senioridade", "gold_genero_cargo",
                   "gold_regiao", "gold_modelo_trabalho"])

assert all(d.n_respondentes.sum() == 14002 for d in (gs, gc, rg, mt)), \
    "as tabelas da gold nao somam 14.002 - reveja a etapa que as gerou"
print(f"gold carregada: {len(gs)}, {len(gc)}, {len(rg)}, {len(mt)} linhas | "
      "as 4 reconciliam com a silver")

## 1. Gênero: quem entra e quem chega ao topo

### 1.1 A participação feminina está caindo

In [ ]:
t = gs.groupby(["ano_pesquisa", "genero"])["n_respondentes"].sum().unstack(fill_value=0)
pct_ano = (t["Feminino"] / t.sum(axis=1) * 100).round(1)

fig, ax = plt.subplots(figsize=(7.5, 3.8))
pct_ano.plot(ax=ax, color=SERIE[0], marker="o", legend=False)
for ano, v in pct_ano.items():
    ax.annotate(f"{v:.1f}%", (ano, v), xytext=(0, 11), textcoords="offset points",
                ha="center", fontsize=11, color=INK, fontweight="semibold")
ax.set(ylim=(0, 32), xticks=ANOS, xlabel="", ylabel="% de mulheres")
moldura(ax, "Participação feminina no mercado de Dados",
        "Queda contínua nos três anos — 2,4 pontos percentuais perdidos")
mostrar(fig, "1_1_participacao_feminina")

tabela(pd.DataFrame({"mulheres": t["Feminino"], "total": t.sum(axis=1),
                     "% feminino": pct_ano}), "Participação por ano")

### 1.2 Quanto mais sênior o cargo, menos mulheres

In [ ]:
gsn = gs[gs.genero.isin(["Masculino", "Feminino"]) & (gs.nivel != NI)]
piv = gsn.pivot_table(index=["nivel", "ano_pesquisa"], columns="genero",
                    values="n_respondentes", aggfunc="sum").fillna(0)
funil = ((piv["Feminino"] / piv.sum(axis=1) * 100)
         .unstack("ano_pesquisa").reindex(ORDEM_NIVEL).round(1))

fig, ax = plt.subplots(figsize=(8.5, 4.2))
funil.plot.bar(ax=ax, color=RAMPA3, width=0.78, rot=0)
ax.bar_label(ax.containers[-1], fmt="%.1f%%", padding=3, fontsize=9.5, color=INK)
ax.axhline(pct_ano.mean(), color=MUTED, lw=1)
ax.set(ylim=(0, 34), xlabel="", ylabel="% de mulheres no nível")
ax.legend(ncol=3, loc="upper right")
moldura(ax, "Mulheres em cada nível de senioridade",
        "O funil estreita em todos os anos — e não melhorou desde 2023")
mostrar(fig, "1_2_funil_senioridade")

tabela(funil, "% feminino por nível e ano")

### 1.3 Em quais cargos elas estão

In [ ]:
CURTO = {"Data Product Manager/ Product Manager (PM/APM/DPM/GPM/PO)": "Product Manager",
         "Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas": "Desenvolvedor / Eng. Software",
         "Engenheiro de Machine Learning/ML Engineer/AI Engineer": "Engenheiro de ML",
         "Engenheiro/Arquiteto de Dados": "Engenheiro / Arquiteto de Dados"}

gcg = gc[gc.genero.isin(["Masculino", "Feminino"]) & (gc.cargo != NI)]
t = gcg.pivot_table(index="cargo", columns="genero", values="n_respondentes",
                  aggfunc="sum").fillna(0)
t["total"] = t["Feminino"] + t["Masculino"]
t = t[t["total"] >= 100]
t["% feminino"] = (t["Feminino"] / t["total"] * 100).round(1)
t.index = [CURTO.get(i, i.split("/")[0][:34]) for i in t.index]
t = t.sort_values("% feminino")

fig, ax = plt.subplots(figsize=(9, 5))
t["% feminino"].plot.barh(ax=ax, color=SERIE[0], width=0.68, legend=False)
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=4, fontsize=9, color=INK2)
ax.axvline(22.9, color=MUTED, lw=1)
ax.set(xlim=(0, 48), xlabel="% de mulheres no cargo", ylabel="")
moldura(ax, "Mulheres em cada cargo",
        "Cargos com ao menos 100 respondentes · linha = média do mercado, 22,9%",
        eixo="x")
mostrar(fig, "1_3_mulheres_por_cargo")

tabela(t.sort_values("% feminino", ascending=False), "Composição por cargo")

### 1.4 A diferença salarial só aparece no topo

In [ ]:
gsal = gs[gs.genero.isin(["Masculino", "Feminino"]) & (gs.nivel != NI)]
w = gsal.groupby(["nivel", "genero"])[["soma_salario", "n_com_salario"]].sum()
sal = ((w.soma_salario / w.n_com_salario).unstack("genero")
       .reindex(ORDEM_NIVEL)[["Feminino", "Masculino"]])
gap = ((sal["Masculino"] - sal["Feminino"]) / sal["Masculino"] * 100).round(1)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
sal.plot.bar(ax=ax, color=[SERIE[0], SERIE[1]], width=0.72, rot=0)
for cont in ax.containers:
    ax.bar_label(cont, fmt="R$ %.0f", padding=3, fontsize=9, color=INK2)
for i, nivel in enumerate(ORDEM_NIVEL):
    ax.annotate(f"gap {gap[nivel]:.1f}%", (i, sal.loc[nivel].max() + 1900),
                ha="center", fontsize=10, color=INK if gap[nivel] > 10 else MUTED,
                fontweight="semibold" if gap[nivel] > 10 else "normal")
ax.set(ylim=(0, 20500), xlabel="", ylabel="salário médio aproximado (R$/mês)")
ax.legend(ncol=2, loc="upper left")
moldura(ax, "Salário médio por gênero e senioridade",
        "Ponto médio das faixas salariais — os três anos somados")
mostrar(fig, "1_4_gap_salarial")

tabela(sal.round(0).assign(**{"gap %": gap}), "Salário por gênero e nível")

## 2. Região: de onde as pessoas trabalham

### 2.1 As regiões estão se aproximando no salário

In [ ]:
r = rg[rg.regiao != NI]
w = r.groupby(["ano_pesquisa", "regiao"])[["soma_salario", "n_com_salario"]].sum()
sal_reg = (w.soma_salario / w.n_com_salario).unstack("regiao")[ORDEM_REGIAO].round(0)

fig, ax = plt.subplots(figsize=(8.5, 4.4))
sal_reg.plot(ax=ax, color=SERIE, marker="o")
ax.set(xticks=ANOS, ylim=(6000, 15000), xlabel="",
       ylabel="salário médio aproximado (R$/mês)")
ax.legend(ncol=5, loc="lower center", fontsize=8.5)
moldura(ax, "Salário médio por região",
        "O Norte sobe 57% em três anos e encosta no Sul e no Centro-oeste")
mostrar(fig, "2_1_salario_por_regiao")

tabela(sal_reg.T, "Salário médio por região e ano")

### 2.2 Parte da diferença é o Sudeste ter mais gente sênior

In [ ]:
comp = (rg[(rg.regiao != NI) & (rg.nivel != NI)]
        .pivot_table(index="regiao", columns="nivel", values="n_respondentes",
                     aggfunc="sum").reindex(ORDEM_REGIAO)[ORDEM_NIVEL])
pct = (comp.div(comp.sum(axis=1), axis=0) * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4))
pct.plot.barh(stacked=True, ax=ax, color=RAMPA3, width=0.66,
              edgecolor=SURFACE, linewidth=1.5)
for i, cont in enumerate(ax.containers):
    ax.bar_label(cont, fmt="%.0f%%", label_type="center", fontsize=9,
                 color="#ffffff" if i == 2 else INK)
ax.invert_yaxis()
ax.set(xlim=(0, 100), xlabel="% dos respondentes da região", ylabel="")
ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.3))
moldura(ax, "Quem é júnior, pleno e sênior em cada região",
        "Sudeste concentra os cargos mais sêniores", eixo="x")
mostrar(fig, "2_2_senioridade_por_regiao")

tabela(pct, "% de cada nível dentro da região")

### 2.3 Em adoção de IA, as regiões empataram

In [ ]:
w = r.groupby(["ano_pesquisa", "regiao"])[["n_usa_ia", "n_com_flag_ia"]].sum()
w = w[w.n_com_flag_ia >= 30]                      # corte de amostra minima
ia = (w.n_usa_ia / w.n_com_flag_ia * 100).unstack("regiao")[ORDEM_REGIAO].round(1)

fig, ax = plt.subplots(figsize=(8.5, 4.4))
ia.plot(ax=ax, color=SERIE, marker="o")
ax.set(xticks=ANOS, ylim=(70, 104), xlabel="",
       ylabel="% que usa IA generativa no trabalho")
ax.legend(ncol=5, loc="lower center", fontsize=8.5)
moldura(ax, "Adoção de IA generativa por região",
        "Um teto comum de ~98% (o Norte sai em 2025 por amostra abaixo de 30)")
mostrar(fig, "2_3_adocao_ia_por_regiao")

tabela(ia.T, "% de adoção de IA por região e ano")

### 2.4 Fora do Sudeste, o remoto é a porta de entrada

In [ ]:
rem = r.groupby("regiao")[["n_remoto", "n_com_modelo"]].sum().reindex(ORDEM_REGIAO)
rem["% remoto"] = (rem.n_remoto / rem.n_com_modelo * 100).round(1)
rem = rem.sort_values("% remoto")

fig, ax = plt.subplots(figsize=(8, 3.8))
rem["% remoto"].plot.barh(ax=ax, color=SERIE[0], width=0.62, legend=False)
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=4, fontsize=10, color=INK)
ax.set(xlim=(0, 68), xlabel="% em modelo 100% remoto", ylabel="")
moldura(ax, "Trabalho remoto por região",
        "Quanto mais longe do Sudeste, maior a dependência do remoto", eixo="x")
mostrar(fig, "2_4_remoto_por_regiao")

tabela(rem, "Remoto por região")

## 3. Modelo de trabalho: como as pessoas trabalham

### 3.1 O remoto está recuando

In [ ]:
m = mt[mt.modelo != NI]
comp = m.pivot_table(index="ano_pesquisa", columns="modelo",
                     values="n_respondentes", aggfunc="sum")[ORDEM_MODELO]
pct_m = (comp.div(comp.sum(axis=1), axis=0) * 100).round(1)

fig, ax = plt.subplots(figsize=(8.5, 4))
pct_m.plot.bar(stacked=True, ax=ax, color=RAMPA4, width=0.52, rot=0,
               edgecolor=SURFACE, linewidth=1.5)
for i, cont in enumerate(ax.containers):
    ax.bar_label(cont, fmt="%.1f%%", label_type="center", fontsize=9,
                 color="#ffffff" if i >= 2 else INK)
ax.set(ylim=(0, 100), xlabel="", ylabel="% dos respondentes do ano")
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.26))
moldura(ax, "Modelo de trabalho por ano",
        "Remoto perde 6,6 pontos; presencial ganha 4,2")
mostrar(fig, "3_1_modelo_por_ano")

tabela(pct_m, "% de cada modelo por ano")

### 3.2 O presencial paga menos, mesmo comparando gente do mesmo nível

In [ ]:
mn = mt[(mt.modelo != NI) & (mt.nivel != NI)]
media = lambda d: (d.soma_salario.sum() / d.n_com_salario.sum())

ingenuo = mn.groupby("modelo").apply(media).reindex(ORDEM_MODELO)
ctrl = (mn.groupby(["nivel", "modelo"]).apply(media)
        .unstack("modelo").reindex(ORDEM_NIVEL)[ORDEM_MODELO].round(0))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4),
                               gridspec_kw={"width_ratios": [1, 2.1]})

ingenuo.plot.bar(ax=ax1, color=RAMPA4, width=0.6, rot=0, legend=False)
ax1.bar_label(ax1.containers[0], fmt="R$ %.0f", padding=3, fontsize=8.5, color=INK2)
ax1.set_xticklabels([x.replace(" ", "\n") for x in ORDEM_MODELO], fontsize=8.5)
ax1.set(ylim=(0, 13500), xlabel="", ylabel="salário médio aproximado (R$/mês)")
moldura(ax1, "Sem separar por senioridade", "Mistura júnior e sênior na mesma conta")

ctrl.plot.bar(ax=ax2, color=RAMPA4, width=0.78, rot=0)
ax2.set(ylim=(0, 18500), xlabel="", ylabel="")
ax2.legend(ncol=4, loc="upper left", fontsize=8.5)
moldura(ax2, "Separando por senioridade", "Cada nível comparado só com ele mesmo")
mostrar(fig, "3_2_modelo_x_salario")

tabela(ctrl, "Salário médio por nível e modelo")

### 3.3 Por que o número cru enganava

In [ ]:
comp = mn.pivot_table(index="modelo", columns="nivel", values="n_respondentes",
                      aggfunc="sum").reindex(ORDEM_MODELO)[ORDEM_NIVEL]
pct_n = (comp.div(comp.sum(axis=1), axis=0) * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 3.6))
pct_n.plot.barh(stacked=True, ax=ax, color=RAMPA3, width=0.62,
                edgecolor=SURFACE, linewidth=1.5)
for i, cont in enumerate(ax.containers):
    ax.bar_label(cont, fmt="%.0f%%", label_type="center", fontsize=9,
                 color="#ffffff" if i == 2 else INK)
ax.invert_yaxis()
ax.set(xlim=(0, 100), xlabel="% dos respondentes do modelo", ylabel="")
ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.36))
moldura(ax, "Quem é júnior, pleno e sênior em cada modelo",
        "O presencial concentra júnior; o remoto concentra sênior", eixo="x")
mostrar(fig, "3_3_senioridade_por_modelo")

tabela(pct_n, "% de cada nível dentro do modelo")

### 3.4 Mulheres dependem mais do remoto

In [ ]:
w = mt[mt.modelo != NI].groupby("modelo")[["n_feminino", "n_com_genero"]].sum()
w = w.reindex(ORDEM_MODELO)
dist = pd.DataFrame({
    "Feminino": w.n_feminino / w.n_feminino.sum() * 100,
    "Masculino": (w.n_com_genero - w.n_feminino) / (w.n_com_genero - w.n_feminino).sum() * 100,
}).round(1)

fig, ax = plt.subplots(figsize=(8.5, 4))
dist.plot.bar(ax=ax, color=[SERIE[0], SERIE[1]], width=0.72, rot=0)
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.1f%%", padding=3, fontsize=9, color=INK2)
ax.set_xticklabels([x.replace(" ", "\n") for x in ORDEM_MODELO], fontsize=9)
ax.set(ylim=(0, 58), xlabel="", ylabel="% dentro do próprio gênero")
ax.legend(ncol=2, loc="upper right")
moldura(ax, "Onde homens e mulheres trabalham",
        "Metade das mulheres trabalha 100% remoto")
mostrar(fig, "3_4_modelo_por_genero")

tabela(dist, "% de cada gênero por modelo")

---
## Resumo da análise


| Evidência | Número |
|---|---|
| Participação feminina em queda | 24,4% → 23,5% → **22,0%** |
| Funil estreitando por senioridade | 28,5% júnior → 20,7% sênior (2025) |
| Diferença salarial concentrada no topo | 5,6% júnior · 1,9% pleno · **15,2% sênior** |
| Remoto como acesso fora do Sudeste | Nordeste 56,9% · Norte 51,2% · Sudeste 41,4% |
| Convergência salarial regional | Norte **+57%** em três anos |
| IA deixou de diferenciar região | ~80% (2023) → **~98%** (2025) |
| Recuada do remoto | 46,3% → **39,7%**; presencial 16,6% → 20,8% |
| Presencial paga menos em todo nível | júnior 3.378 vs 4.254 · sênior 10.241 vs 15.671 |

Os 12 gráficos ficam em `s3://tech-challenge-014478672967/artefatos/frente3_diversidade_loc_modalidade/`.